# 02 - Zone boundaries and NO2 monitors

The feasibility check counts monitors **within 5 km of an approximate city centre**,
then classifies each one against the **zone boundary**. These maps show both, so you
can judge the two definitions side by side:

- orange area: the charging zone (`data/reference/zone_boundaries.geojson`); Portsmouth has none
- dashed circle: 5 km from the centre in `zones.csv`; `+` marks that centre
- **blue** markers: monitors within 5 km; **grey**: further away (shown up to `VIEW_KM`)
- **filled** = usable (>= 3 years with >= 75% capture before the charging year and >= 1 after);
  **hollow** = not usable (too few years, or closed before 2010)
- shape = site type: circle roadside, square background, triangle other (industrial, rural, unknown)

Data: the openair files cached by `scripts/feasibility.py` (AURN, SAQN, LMAM; AQE is offline).
Run that script first if `data/raw/openair/` is empty.

In [ ]:
from pathlib import Path

import folium
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from shapely.geometry import mapping

from caz import feasibility as f
from caz.boundaries import locate, read_boundaries, to_local_metres
from caz.geo import haversine_km
from caz.openair import (
    NETWORKS,
    DownloadError,
    annual_summary_url,
    download,
    metadata_url,
    read_annual_summary,
    read_metadata,
)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw" / "openair"
REFERENCE = ROOT / "data" / "reference"

VIEW_KM = 7.0  # show monitors up to this distance from the centre
YEARS = range(2010, 2026)

# Colours: reference data-viz palette slots 1-2, plus neutral greys
BLUE, ORANGE, GREY, INK = "#2a78d6", "#eb6834", "#8a8984", "#52514e"
SHAPES = {"roadside": "o", "background": "s"}  # anything else: "^"

## Load zones, boundaries and monitors

Same steps as the feasibility script: NO2 sites from all available networks, the same
instrument in two networks merged, annual data pooled, usable years at >= 75% capture.

In [ ]:
zones = pd.read_csv(REFERENCE / "zones.csv", parse_dates=["charging_start", "decision_date"])
boundaries = read_boundaries(REFERENCE / "zone_boundaries.geojson")

meta_parts, annual_parts = [], []
for name, net in NETWORKS.items():
    try:
        meta_parts.append(
            read_metadata(download(metadata_url(net), RAW / name / "metadata.RData"), net)
        )
    except DownloadError:
        print(f"{name}: unavailable, skipped")
        continue
    for year in YEARS:
        try:
            path = download(annual_summary_url(net, year), RAW / name / f"annual_{year}.rds")
        except DownloadError:
            continue
        annual_parts.append(read_annual_summary(path, name))

sites, mapping_table = f.merge_duplicate_sites(f.no2_sites(pd.concat(meta_parts)), list(NETWORKS))
annual = f.pool_merged_annual(pd.concat(annual_parts, ignore_index=True), mapping_table)
years_by_site = f.usable_years(annual).set_index(["network", "site_id"])["usable_years"]
print(len(sites), "NO2 sites")

In [ ]:
def monitors_near(zone: pd.Series) -> pd.DataFrame:
    """Every NO2 site within VIEW_KM of the zone centre, with distance, usability and location."""
    lat0, lon0 = zone["approx_centre_lat"], zone["approx_centre_lon"]
    d = sites.apply(lambda s: haversine_km(lat0, lon0, s["latitude"], s["longitude"]), axis=1)
    near = sites[d <= VIEW_KM].assign(distance_km=d[d <= VIEW_KM].round(2)).copy()
    charging = zone["charging_start"].date()
    keys = zip(near["network"], near["site_id"], strict=True)
    near["usable_years"] = [years_by_site.get(k, []) for k in keys]
    near["usable"] = near["usable_years"].map(lambda y: f.is_usable(y, charging, charging))
    near["within_5km"] = near["distance_km"] <= f.SEARCH_RADIUS_KM
    located = [
        locate(la, lo, boundaries.get(zone["zone_id"]))
        for la, lo in zip(near["latitude"], near["longitude"], strict=True)
    ]
    near["zone_location"] = [loc for loc, _ in located]
    near["distance_to_zone_m"] = [None if m is None else round(m) for _, m in located]
    # flat x/y in km around the centre (east, north), for plotting
    x = (near["longitude"] - lon0) * 111.195 * np.cos(np.radians(lat0))
    y = (near["latitude"] - lat0) * 111.195
    return near.assign(x_km=x.values, y_km=y.values).sort_values("distance_km")


monitors = {z["zone_id"]: monitors_near(z) for _, z in zones.iterrows()}
pd.DataFrame(
    {
        zid: {
            "within 5 km": int(m["within_5km"].sum()),
            "usable within 5 km": int((m["within_5km"] & m["usable"]).sum()),
            "usable inside zone": int(((m["zone_location"] == "inside") & m["usable"]).sum()),
            f"usable 5-{VIEW_KM:g} km": int((~m["within_5km"] & m["usable"]).sum()),
        }
        for zid, m in monitors.items()
    }
).T

## Overview: every zone at the same scale

Each panel is 2 x `VIEW_KM` wide, so zone sizes can be compared directly.

In [ ]:
def draw_zone(ax, zone: pd.Series, half_width_km: float, label: bool) -> None:
    """Boundary, 5 km circle, centre and monitors for one zone, in km around the centre."""
    zid = zone["zone_id"]
    lat0, lon0 = zone["approx_centre_lat"], zone["approx_centre_lon"]
    if zid in boundaries:
        local = to_local_metres(boundaries[zid], lat0, lon0)
        for poly in getattr(local, "geoms", [local]):
            xs, ys = np.array(poly.exterior.coords).T / 1000
            ax.fill(xs, ys, color=ORANGE, alpha=0.18, lw=0)
            ax.plot(xs, ys, color=ORANGE, lw=1.5)
    else:
        ax.text(
            0.5,
            0.04,
            "no boundary available",
            transform=ax.transAxes,
            ha="center",
            color=INK,
            fontsize=8,
        )
    ax.add_patch(plt.Circle((0, 0), f.SEARCH_RADIUS_KM, fill=False, ls="--", lw=1, color=GREY))
    ax.plot(0, 0, "+", color=INK, ms=8, mew=1.2)

    m = monitors[zid]
    for _, s in m.iterrows():
        colour = BLUE if s["within_5km"] else GREY
        ax.plot(
            s["x_km"],
            s["y_km"],
            SHAPES.get(s["site_class"], "^"),
            ms=8,
            mfc=colour if s["usable"] else "white",
            mec=colour if s["usable"] else colour,
            mew=1.5,
            zorder=3,
        )
        if label and s["usable"]:
            ax.annotate(
                s["site_id"],
                (s["x_km"], s["y_km"]),
                xytext=(5, 4),
                textcoords="offset points",
                fontsize=8,
                color=INK,
            )
    ax.set_xlim(-half_width_km, half_width_km)
    ax.set_ylim(-half_width_km, half_width_km)
    ax.set_aspect("equal")
    ax.tick_params(labelsize=7, colors=INK)
    for side in ax.spines.values():
        side.set_color("#d4d3cf")
    n_in = int(((m["zone_location"] == "inside") & m["usable"]).sum())
    n_5 = int((m["within_5km"] & m["usable"]).sum())
    inside = f"{n_in} inside" if zid in boundaries else "? inside"
    ax.set_title(
        f"{zone['city']} ({zone['class']}): usable {n_5} within 5 km, {inside}",
        fontsize=9,
        color="#0b0b0b",
    )


LEGEND = [
    Patch(facecolor=ORANGE, alpha=0.35, edgecolor=ORANGE, label="charging zone"),
    Line2D([], [], ls="--", color=GREY, label="5 km from centre"),
    Line2D([], [], ls="", marker="o", ms=8, color=BLUE, label="usable, within 5 km"),
    Line2D(
        [],
        [],
        ls="",
        marker="o",
        ms=8,
        mfc="white",
        mec=BLUE,
        mew=1.5,
        label="not usable, within 5 km",
    ),
    Line2D([], [], ls="", marker="o", ms=8, color=GREY, label="usable, beyond 5 km"),
    Line2D(
        [],
        [],
        ls="",
        marker="o",
        ms=8,
        mfc="white",
        mec=GREY,
        mew=1.5,
        label="not usable, beyond 5 km",
    ),
    Line2D([], [], ls="", marker="o", ms=7, color=INK, label="roadside"),
    Line2D([], [], ls="", marker="s", ms=7, color=INK, label="background"),
    Line2D([], [], ls="", marker="^", ms=7, color=INK, label="other"),
]

fig, axes = plt.subplots(3, 4, figsize=(16, 12.5), layout="constrained")
for ax, (_, zone) in zip(axes.flat, zones.iterrows(), strict=False):
    draw_zone(ax, zone, VIEW_KM, label=False)
legend_ax = axes.flat[-1]
legend_ax.axis("off")
legend_ax.legend(handles=LEGEND, loc="center", frameon=False, fontsize=9)
fig.supxlabel("km east of centre", color=INK)
fig.supylabel("km north of centre", color=INK)
plt.show()

## Zoom on one zone

Labels show the site code of every usable monitor. Change `ZONE` and `HALF_WIDTH_KM`.
The table below the map lists every monitor in view, nearest first.

Thin strips and small notches on some boundaries are real: zones such as Bristol include
stretches of approach roads, and boundaries follow road edges into junctions. They are
under 0.3% of any zone's area.

In [ ]:
ZONE = "bradford"
HALF_WIDTH_KM = 3.0

zone = zones.set_index("zone_id").loc[ZONE].copy()
zone["zone_id"] = ZONE
fig, ax = plt.subplots(figsize=(8, 8), layout="constrained")
draw_zone(ax, zone, HALF_WIDTH_KM, label=True)
ax.legend(handles=LEGEND, loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=8)
ax.set_xlabel("km east of centre", color=INK)
ax.set_ylabel("km north of centre", color=INK)
plt.show()

cols = [
    "network",
    "site_id",
    "site_name",
    "site_class",
    "distance_km",
    "within_5km",
    "zone_location",
    "distance_to_zone_m",
    "usable",
    "usable_years",
]
monitors[ZONE][cols]

## Interactive map with streets

Zoom and click markers for details. The basemap is OpenStreetMap (© OpenStreetMap contributors).
Markers use the same colours and filled/hollow rule as above (shape is not available here;
the popup gives the site type).

In [ ]:
def interactive_map(zone_id: str) -> folium.Map:
    z = zones.set_index("zone_id").loc[zone_id]
    centre = [z["approx_centre_lat"], z["approx_centre_lon"]]
    fmap = folium.Map(location=centre, zoom_start=13, tiles="OpenStreetMap")
    if zone_id in boundaries:
        folium.GeoJson(
            mapping(boundaries[zone_id]),
            name="zone",
            style_function=lambda _: {"color": ORANGE, "weight": 2, "fillOpacity": 0.15},
        ).add_to(fmap)
    folium.Circle(
        centre, radius=f.SEARCH_RADIUS_KM * 1000, color=GREY, dash_array="6", fill=False
    ).add_to(fmap)
    for _, s in monitors[zone_id].iterrows():
        colour = BLUE if s["within_5km"] else GREY
        years = ", ".join(map(str, s["usable_years"])) or "none"
        popup = (
            f"<b>{s['site_name']}</b> ({s['network']}:{s['site_id']})<br>{s['site_class']}, "
            f"{s['distance_km']} km from centre, {s['zone_location']}<br>"
            f"usable: {'yes' if s['usable'] else 'no'}<br>usable years: {years}"
        )
        folium.CircleMarker(
            [s["latitude"], s["longitude"]],
            radius=7,
            color=colour,
            weight=2,
            fill=True,
            fill_color=colour if s["usable"] else "white",
            fill_opacity=1,
            popup=folium.Popup(popup, max_width=300),
            tooltip=f"{s['site_id']} - {s['site_name']}",
        ).add_to(fmap)
    return fmap


interactive_map(ZONE)